# [Lab1] 데이터 전처리 with SageMaker Processing

이 노트북에서는 SageMaker Processing Job을 사용하여 은행 마케팅 데이터를 전처리합니다.

## 주요 내용
- SageMaker Processing Job 설정
- 데이터 전처리 스크립트 작성
- 전처리된 데이터를 S3에 저장

## 1. 환경 설정 및 변수 로드

In [1]:
# 이전 노트북에서 저장한 변수들 로드
%store -r

# [2026-06-20 수정] 0-setup.ipynb 를 먼저 실행하지 않으면 %store 복원이 비어 NameError 가 납니다.
_required = ['bucket', 'prefix', 'region']
_missing = [v for v in _required if v not in globals()]
if _missing:
    raise RuntimeError(
        f"필수 변수 누락: {_missing}\n"
        "👉 같은 JupyterLab 커널에서 0-setup.ipynb 를 먼저 실행해주세요."
    )

print("✅ 저장된 변수들을 로드했습니다.")
print(f"   - S3 버킷: {bucket}")
print(f"   - S3 프리픽스: {prefix}")
print(f"   - 리전: {region}")# 이전 노트북에서 저장한 변수들 로드
%store -r

# [2026-06-20 수정] 0-setup.ipynb 를 먼저 실행하지 않으면 %store 복원이 비어 NameError 가 납니다.
_required = ['bucket', 'prefix', 'region']
_missing = [v for v in _required if v not in globals()]
if _missing:
    raise RuntimeError(
        f"필수 변수 누락: {_missing}\n"
        "👉 같은 JupyterLab 커널에서 0-setup.ipynb 를 먼저 실행해주세요."
    )

print("✅ 저장된 변수들을 로드했습니다.")
print(f"   - S3 버킷: {bucket}")
print(f"   - S3 프리픽스: {prefix}")
print(f"   - 리전: {region}")

✅ 저장된 변수들을 로드했습니다.
   - S3 버킷: csv-file-store-db634350
   - S3 프리픽스: dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm
   - 리전: us-west-2
✅ 저장된 변수들을 로드했습니다.
   - S3 버킷: csv-file-store-db634350
   - S3 프리픽스: dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm
   - 리전: us-west-2


In [2]:
# 필수 라이브러리 임포트
import sagemaker
import boto3
import os
from time import gmtime, strftime
from sagemaker.sklearn import SKLearn
from sagemaker.processing import FrameworkProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker_studio import Project

# AWS 세션 초기화
boto_session = boto3.Session()
sess = sagemaker.Session()

print("✅ 라이브러리 임포트 및 세션 초기화 완료")

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
✅ 라이브러리 임포트 및 세션 초기화 완료


## 2. 전처리 스크립트 준비

SageMaker Processing Job에서 실행될 전처리 스크립트를 작성합니다.

In [3]:
# 처리 작업을 위한 디렉토리 생성
!mkdir -p processing/requirements

print("✅ 처리 작업 디렉토리 생성 완료")

✅ 처리 작업 디렉토리 생성 완료


In [4]:
%%writefile processing/requirements/requirements.txt
pandas
numpy
scikit-learn

Overwriting processing/requirements/requirements.txt


In [5]:
%%writefile processing/preprocessing.py
import pandas as pd
import numpy as np
import argparse
import os
import subprocess
import sys
from sklearn.model_selection import train_test_split  # [2026-06-20] 추가: 층화 분할용

# Requirements 설치
def install_requirements():
    requirements_path = '/opt/ml/processing/input/code/requirements.txt'
    if os.path.exists(requirements_path):
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', requirements_path])
            print("Requirements installed successfully")
        except subprocess.CalledProcessError as e:
            print(f"Error installing requirements: {e}")
            # 필수 패키지만 개별 설치
            essential_packages = ['pandas', 'numpy', 'scikit-learn']
            for package in essential_packages:
                try:
                    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
                except Exception as e:
                    # [2026-06-20] 수정: bare except 제거 — 실패 원인을 출력해 디버깅 가능하도록 변경
                    print(f"Failed to install {package}: {e}")
    else:
        print(f"Requirements file not found at {requirements_path}")

# Requirements 설치 실행
install_requirements()

def _parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--filepath', type=str, default='/opt/ml/processing/input/')
    parser.add_argument('--filename', type=str, default='bank-additional-full.csv')
    parser.add_argument('--outputpath', type=str, default='/opt/ml/processing/output/')
    return parser.parse_known_args()

def process_data(args):
    """데이터 전처리 함수"""
    # 데이터 로드
    df = pd.read_csv(os.path.join(args.filepath, args.filename))
    print(f"원본 데이터 크기: {df.shape}")
    
    # 데이터 전처리
    # 1. 점(.)을 언더스코어(_)로 변경 (예: 'admin.' -> 'admin', 'basic.4y' -> 'basic_4y')
    # [2026-06-20] 수정: 값 치환을 문자열(object) 컬럼으로 한정 — 숫자 컬럼/컬럼명에는 영향 없음을 명시
    obj_cols = df.select_dtypes(include='object').columns
    df[obj_cols] = df[obj_cols].replace(regex=r'\.', value='_')
    df[obj_cols] = df[obj_cols].replace(regex=r'\_$', value='')
    
    # 2. 새로운 특성 추가
    df["no_previous_contact"] = (df["pdays"] == 999).astype(int)
    df["not_working"] = df["job"].isin(["student", "retired", "unemployed"]).astype(int)
    
    # 3. 불필요한 컬럼 제거
    df = df.drop(['duration', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed'], axis=1)
    
    # 4. 범주형 변수 원-핫 인코딩
    # [2026-06-20] 수정: pandas 2.x 의 get_dummies 는 bool(True/False)을 반환하여
    #   CSV 에 'True'/'False' 가 기록되고 XGBoost CSV 파싱이 깨짐 → .astype(int) 로 0/1 보장
    df = pd.get_dummies(df).astype(int)
    print(f"전처리 후 데이터 크기: {df.shape}")
    
    # 5. 훈련/검증/테스트 데이터 분할 (70/20/10)
    # [2026-06-20] 수정: 불균형 타깃(~11% 양성)의 클래스 비율을 train/val/test 에서 유지하도록
    #   단순 셔플 분할(np.split) 대신 층화 분할(stratify)로 변경
    train_data, temp_data = train_test_split(
        df, test_size=0.3, random_state=42, stratify=df['y_yes']
    )
    validation_data, test_data = train_test_split(
        temp_data, test_size=1/3, random_state=42, stratify=temp_data['y_yes']
    )
    
    print(f"훈련 데이터: {train_data.shape}")
    print(f"검증 데이터: {validation_data.shape}")
    print(f"테스트 데이터: {test_data.shape}")
    
    return train_data, validation_data, test_data, df

def save_data(train_data, validation_data, test_data, df, output_path):
    """전처리된 데이터 저장"""
    # 출력 디렉토리 생성
    for subdir in ['train', 'validation', 'test', 'baseline']:
        os.makedirs(os.path.join(output_path, subdir), exist_ok=True)
    
    # 훈련 데이터 저장 (타겟 변수를 첫 번째 컬럼으로)
    pd.concat([train_data['y_yes'], train_data.drop(['y_yes','y_no'], axis=1)], axis=1).to_csv(
        os.path.join(output_path, 'train/train.csv'), index=False, header=False)
    
    # 검증 데이터 저장
    pd.concat([validation_data['y_yes'], validation_data.drop(['y_yes','y_no'], axis=1)], axis=1).to_csv(
        os.path.join(output_path, 'validation/validation.csv'), index=False, header=False)
    
    # 테스트 데이터 저장 (X, y 분리)
    test_data['y_yes'].to_csv(os.path.join(output_path, 'test/test_y.csv'), index=False, header=False)
    test_data.drop(['y_yes','y_no'], axis=1).to_csv(
        os.path.join(output_path, 'test/test_x.csv'), index=False, header=False)
    
    # 베이스라인 데이터 저장
    baseline_path = os.path.join(output_path, 'baseline/baseline.csv')
    df.drop(['y_yes','y_no'], axis=1).to_csv(baseline_path, index=False, header=False)
    
    return baseline_path

if __name__=="__main__":
    args, _ = _parse_args()
    train_data, validation_data, test_data, df = process_data(args)
    save_data(train_data, validation_data, test_data, df, args.outputpath)
    print("✅ 데이터 전처리 완료")

Overwriting processing/preprocessing.py


## 3. 입력 및 출력 경로 설정

SageMaker Processing Job에서 사용할 입력 데이터와 출력 경로를 설정합니다.

In [6]:
# 입력 데이터를 S3에 업로드
input_source = sess.upload_data(
    './bank-additional/bank-additional-full.csv', 
    bucket=bucket, 
    key_prefix=f'{prefix}/input_data'
)

print(f"✅ 입력 데이터 업로드 완료: {input_source}")

✅ 입력 데이터 업로드 완료: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/input_data/bank-additional-full.csv


In [7]:
# 출력 경로 설정
train_path = f"s3://{bucket}/{prefix}/train"
validation_path = f"s3://{bucket}/{prefix}/validation"
test_path = f"s3://{bucket}/{prefix}/test"
baseline_path = f"s3://{bucket}/{prefix}/baseline"

print("✅ 출력 경로 설정 완료:")
print(f"   - 훈련 데이터: {train_path}")
print(f"   - 검증 데이터: {validation_path}")
print(f"   - 테스트 데이터: {test_path}")
print(f"   - 베이스라인: {baseline_path}")

✅ 출력 경로 설정 완료:
   - 훈련 데이터: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/train
   - 검증 데이터: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/validation
   - 테스트 데이터: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/test
   - 베이스라인: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/baseline


## 4. 프로젝트 설정

Processing Job 실행에 필요한 프로젝트 정보(IAM 역할 등)를 가져옵니다.
(전처리 단계는 MLflow 로깅을 하지 않습니다 — 실험 추적은 2번 훈련 노트북부터 시작합니다.)

In [8]:
# (전처리 단계에서는 MLflow run 을 사용하지 않습니다.)

In [9]:
# 프로젝트 설정 (Processing Job 에 필요한 IAM 역할/도메인 정보)
project = Project()
role = project.iam_role
domain_id = project.domain_id
project_id = project.id

print("✅ 프로젝트 설정 완료")
print(f"   - IAM 역할: {role}")

✅ 프로젝트 설정 완료
   - IAM 역할: arn:aws:iam::975050309668:role/datazone_usr_role_4gwjxwgbrut11s_3x3ldzh5p3u80g


## 5. SageMaker Processing Job 실행

설정된 전처리 스크립트를 SageMaker Processing Job으로 실행합니다.

In [10]:
# SageMaker Processing Job 설정
sklearn_processor = FrameworkProcessor(
    estimator_cls=SKLearn,
    framework_version="1.2-1",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1, 
    base_job_name='bank-data-preprocessing'
)

print("✅ Processing Job 프로세서 설정 완료")

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.ProcessingJob.NetworkConfig.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.ProcessingJob.NetworkConfig.VpcConfig.SecurityGroupIds
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.SecurityGroupIds
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
✅ Processing Job 프로세서 설정 완료


In [11]:
# 입력 및 출력 설정
processing_inputs = [
    ProcessingInput(
        source=input_source, 
        destination="/opt/ml/processing/input",
        s3_input_mode="File",
        # [2026-06-20] 수정: 단일 S3 객체 + 단일 인스턴스에서 ShardedByS3Key 는 무의미하며
        #   인스턴스 수 증가 시 한 인스턴스만 데이터를 받는 오해를 유발 → FullyReplicated 로 변경
        s3_data_distribution_type="FullyReplicated"
    )
]

processing_outputs = [
    ProcessingOutput(
        output_name="train_data", 
        source="/opt/ml/processing/output/train",
        destination=train_path,
    ),
    ProcessingOutput(
        output_name="validation_data", 
        source="/opt/ml/processing/output/validation", 
        destination=validation_path
    ),
    ProcessingOutput(
        output_name="test_data", 
        source="/opt/ml/processing/output/test", 
        destination=test_path
    ),
    ProcessingOutput(
        output_name="baseline_data", 
        source="/opt/ml/processing/output/baseline", 
        destination=baseline_path
    )
]

print("✅ 입력/출력 설정 완료")

✅ 입력/출력 설정 완료


In [12]:
# Processing Job 실행
print("🚀 SageMaker Processing Job 시작...")

sklearn_processor.run(
    inputs=processing_inputs,
    code='processing/preprocessing.py',
    outputs=processing_outputs,
    dependencies=['processing/requirements/requirements.txt'],
    wait=True
)

print("✅ Processing Job 완료!")

🚀 SageMaker Processing Job 시작...
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.VpcConfig.SecurityGroupIds
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
....................CodeArtifact repository not specified. Skipping login.
Found existing installation: typing 3.7.4.3
Uninstalling typing-3.7.4.3:
  Successfully uninstalled typing-3.7.4.3
[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Requirements installed successfully
원본 데이터 크기: (41188, 21)
전처리 후 데이터 크기: (41188, 61)
훈련 데이터: (28831, 61)
검증 데이터: (8238, 61)
테스트 데이터: (4119, 61)
✅ 데이터 전처리 완료

✅ Processing Job 완료!


## 6. 결과 확인

전처리된 데이터가 올바르게 생성되었는지 확인합니다.

In [13]:
import pandas as pd
import io

# 훈련 데이터 확인
train_data = sess.read_s3_file(
    bucket=bucket,
    key_prefix=f"{prefix}/train/train.csv"
)

df_train = pd.read_csv(io.StringIO(train_data), header=None)

print("✅ 전처리 결과 확인:")
print(f"   - 훈련 데이터 크기: {df_train.shape}")
print(f"   - 첫 번째 컬럼 (타겟): {df_train[0].value_counts()}")

print("\n📊 훈련 데이터 미리보기:")
display(df_train.head())

✅ 전처리 결과 확인:
   - 훈련 데이터 크기: (28831, 60)
   - 첫 번째 컬럼 (타겟): 0
0    25583
1     3248
Name: count, dtype: int64

📊 훈련 데이터 미리보기:


,0,1,2,3,4,5,6,7,8,9,...,50,51,52,53,54,55,56,57,58,59
0,0,50,2,999,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,1,0
1,0,51,5,999,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
2,0,46,2,999,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
3,0,46,1,999,0,1,0,1,0,0,...,0,0,0,1,0,0,0,0,1,0
4,1,25,5,999,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0


In [14]:
# Processing Job 정보 확인
print("✅ 전처리 완료")
print(f"   - Processing Job: {sklearn_processor.latest_job.name}")

✅ 전처리 완료
   - Processing Job: bank-data-preprocessing-2026-06-21-14-00-36-379


In [15]:
# 다음 노트북에서 사용할 변수들 저장
%store input_source
%store train_path
%store validation_path
%store test_path
%store baseline_path

print("✅ 변수 저장 완료")
print("\n📋 저장된 경로:")
print(f"   - 입력 데이터: {input_source}")
print(f"   - 훈련 데이터: {train_path}")
print(f"   - 검증 데이터: {validation_path}")
print(f"   - 테스트 데이터: {test_path}")
print(f"   - 베이스라인: {baseline_path}")

Stored 'input_source' (str)
Stored 'train_path' (str)
Stored 'validation_path' (str)
Stored 'test_path' (str)
Stored 'baseline_path' (str)
✅ 변수 저장 완료

📋 저장된 경로:
   - 입력 데이터: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/input_data/bank-additional-full.csv
   - 훈련 데이터: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/train
   - 검증 데이터: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/validation
   - 테스트 데이터: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/test
   - 베이스라인: s3://csv-file-store-db634350/dzd-dk65fuq5tsejww/4gwjxwgbrut11s/dev/sagemaker/DEMO-xgboost-dm/baseline


## ✅ 데이터 전처리 완료

SageMaker Processing Job을 사용한 데이터 전처리가 성공적으로 완료되었습니다!

### 완료된 작업
- ✅ 원본 데이터 전처리 (특성 엔지니어링, 원-핫 인코딩)
- ✅ 훈련/검증/테스트 데이터 분할
- ✅ 전처리된 데이터를 S3에 저장

### 다음 단계
이제 `2-training.ipynb` 노트북으로 진행하여 전처리된 데이터로 모델을 훈련할 수 있습니다.

### 생성된 데이터
- **훈련 데이터**: 모델 훈련용 (70%)
- **검증 데이터**: 하이퍼파라미터 튜닝용 (20%)
- **테스트 데이터**: 최종 모델 평가용 (10%)
- **베이스라인**: 모델 모니터링용